# Posterior Collapse Dial

Chapter 16: Variational Inference and Latent Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

See how increasing beta can shrink latent usage toward the prior in a toy VAE objective.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-16/posterior-collapse/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's defaults: observation x, decoder noise sx, KL weight beta.
# Toy objective in the latent mean m: -(m - x)^2 / (2 sx^2) - beta m^2 / 2
x, sx, beta = 1.8, 0.9, 1.5


def objective(m, beta):
    return -0.5 * (m - x) ** 2 / sx ** 2 - 0.5 * beta * m ** 2


m_star = x / (1 + beta * sx ** 2)
print(f"optimal latent mean m* = x / (1 + beta sx^2) = {m_star:.4f}")

# ── Check by gradient ascent ────────────────────────────────────────────────
m = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([m], lr=0.1)
for _ in range(500):
    opt.zero_grad()
    (-objective(m, beta)).backward()
    opt.step()
print("gradient ascent m =", m.item())

# ── Raising beta pushes the latent toward the prior mean 0 ──────────────────
print("\n beta    m*       fraction of x the latent still carries")
for b in (0.0, 0.5, 1.5, 3.0, 6.0, 30.0):
    ms = x / (1 + b * sx ** 2)
    print(f"{b:5.1f}  {ms:.4f}   {ms / x:.3f}")

# A noisier decoder (bigger sx) collapses sooner: the reconstruction term is weaker
print("\n sx    m* at beta = 1.5")
for s in (0.3, 0.9, 2.0):
    print(f"{s:4.1f}   {x / (1 + beta * s ** 2):.4f}")

optimal latent mean m* = x / (1 + beta sx^2) = 0.8126


gradient ascent m = 0.8126410835214446

 beta    m*       fraction of x the latent still carries
  0.0  1.8000   1.000
  0.5  1.2811   0.712
  1.5  0.8126   0.451
  3.0  0.5248   0.292
  6.0  0.3072   0.171
 30.0  0.0711   0.040

 sx    m* at beta = 1.5
 0.3   1.5859
 0.9   0.8126
 2.0   0.2571
